# GraphNLI-AXb on Google Colab: ALBERT XXLarge

Fine-tune `albert/albert-xxlarge-v2` for an initial `pair_cls` versus `piig_v3` comparison. Select a GPU runtime before running the notebook. This checkpoint is English-only, so the default evaluation is AX-b plus XNLI English.

In [ ]:
import platform
import subprocess

print('Python:', platform.python_version())
subprocess.run(['nvidia-smi'], check=False)
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
except ImportError:
    print('Torch is not installed yet.')

## Project source

Set `REPO_URL` after publishing the repository, or leave it empty and upload a ZIP containing the `GraphNLI-AXb` directory.

In [ ]:
from pathlib import Path

REPO_URL = ''  # Example: https://github.com/<user>/GraphNLI-AXb.git
PROJECT_NAME = 'GraphNLI-AXb'
CONTENT_DIR = Path('/content')
PROJECT_DIR = CONTENT_DIR / PROJECT_NAME

if PROJECT_DIR.exists():
    raise RuntimeError(f'{PROJECT_DIR} already exists. Restart the Colab runtime or choose a new PROJECT_NAME.')

In [ ]:
import shutil

if REPO_URL:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
else:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Upload exactly one ZIP archive of the project directory.')
    archive_name = next(iter(uploaded))
    archive_path = CONTENT_DIR / archive_name
    shutil.unpack_archive(str(archive_path), str(CONTENT_DIR))
    if not PROJECT_DIR.exists():
        candidates = [path for path in CONTENT_DIR.iterdir() if path.is_dir() and (path / 'pyproject.toml').exists()]
        if len(candidates) != 1:
            raise RuntimeError('Archive must contain one project directory with pyproject.toml.')
        PROJECT_DIR = candidates[0]

print('Project directory:', PROJECT_DIR)
assert (PROJECT_DIR / 'pyproject.toml').exists()

## Poetry environment

Recent Colab runtimes can use Python 3.13, while this project targets Python 3.11-3.12. The next cell uses `uv` to install Python 3.11 and creates an isolated Poetry environment with it. `sentencepiece` is installed because ALBERT tokenization requires it.

In [ ]:
%pip install -q poetry uv
subprocess.run(['uv', 'python', 'install', '3.11'], check=True)
python311 = subprocess.check_output(['uv', 'python', 'find', '3.11'], text=True).strip()
print('Poetry interpreter:', python311)
# Override the global setting used by earlier notebook revisions.
subprocess.run(['poetry', 'config', 'virtualenvs.create', 'true', '--local'], cwd=PROJECT_DIR, check=True)
subprocess.run(['poetry', 'config', 'virtualenvs.in-project', 'true', '--local'], cwd=PROJECT_DIR, check=True)
subprocess.run(['poetry', 'env', 'use', python311], cwd=PROJECT_DIR, check=True)
subprocess.run(['poetry', 'install', '--no-interaction', '--no-ansi'], cwd=PROJECT_DIR, check=True)
subprocess.run(['poetry', 'run', 'pip', 'install', '-q', 'sentencepiece'], cwd=PROJECT_DIR, check=True)
subprocess.run(['poetry', 'run', 'python', '-c', 'import torch, transformers; print(torch.__version__, transformers.__version__)'], cwd=PROJECT_DIR, check=True)

## Datasets

Downloads the official AX-b, XNLI, and MultiNLI archives. MultiNLI is required by the reference-compatible protocol.

In [ ]:
subprocess.run(
    ['poetry', 'run', 'python', 'scripts/download_datasets.py', '--with-mnli'],
    cwd=PROJECT_DIR,
    check=True,
)

## Training configuration

Start with `BATCH_SIZE=2` on a 16 GiB T4. Try `4` on a 24 GiB L4, and increase to `8` only after a successful short run on a 40 GiB A100-class GPU. This codebase does not yet use mixed-precision training, so these conservative values avoid CUDA out-of-memory failures.

In [ ]:
import os
import torch

MODEL_NAME = 'albert/albert-xxlarge-v2'
MODELS = 'pair_cls,piig_v3'
BATCH_SIZE = 2  # T4-safe starting point; set 8 only with sufficient VRAM.
TRAIN_LIMIT = 2000
EPOCHS = 6
LEARNING_RATE = '1e-5'
SEEDS = '42'

assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime > Change runtime type > T4/L4/A100 GPU.'
vram_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
if BATCH_SIZE >= 8 and vram_gib < 24:
    print(f'WARNING: batch size {BATCH_SIZE} is likely to exhaust {vram_gib:.1f} GiB of VRAM with ALBERT XXLarge. Use 1-2 on a T4, or request an L4/A100 runtime.')

In [ ]:
command = [
    'poetry', 'run', 'graphnli-train',
    '--models', MODELS,
    '--seeds', SEEDS,
    '--model-name', MODEL_NAME,
    '--training-protocol', 'mnli',
    '--train-limit', str(TRAIN_LIMIT),
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
    '--lr', LEARNING_RATE,
    '--selection-metric', 'nli3/macro_f1',
    '--patience', '2',
    '--eval-languages', 'en',
    '--device', 'cuda',
    '--output-dir', 'runs',
]
print(' '.join(command))
log_path = PROJECT_DIR / 'colab_training.log'
with log_path.open('w', encoding='utf-8') as log_file:
    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'},
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
    return_code = process.wait()
if return_code:
    raise RuntimeError(f'Training failed with exit code {return_code}. Full log: {log_path}')

## Inspect and download results

Each invocation creates an incremental experiment directory. PIIG v3 explanation SVGs and JSON files are under `piig_v3/seed_42/explanations/axb/`.

In [ ]:
# Reopen the final portion of the streamed training log when needed.
if 'log_path' not in globals():
    log_path = PROJECT_DIR / 'colab_training.log'
if log_path.exists():
    print(''.join(log_path.read_text(encoding='utf-8').splitlines(keepends=True)[-100:]))
else:
    print(f'No log found at {log_path}. Run the training cell first.')

In [ ]:
import json

experiments = sorted((PROJECT_DIR / 'runs').glob('experiment_*'))
experiment_dir = experiments[-1]
summary = json.loads((experiment_dir / 'summary.json').read_text())
for result in summary:
    axb = result['eval']['axb']
    print(result['variant'], {key: round(axb[key], 4) for key in ['binary/mcc', 'binary/macro_f1', 'binary/accuracy', 'binary/ece', 'binary/brier']})
print('Experiment:', experiment_dir)
print('PIIG graphs:', experiment_dir / 'piig_v3' / 'seed_42' / 'explanations' / 'axb')

In [ ]:
from google.colab import files

archive = shutil.make_archive(str(experiment_dir), 'zip', root_dir=experiment_dir)
files.download(archive)